# Module 2 — Data Cleaning

**Modular Daraz Analysis — Module 2 of 6**

**Purpose.** Convert the raw marketplace text into analysis-ready columns, remove repeated listings, validate the result, and save a reusable cleaned CSV.

**Course concepts visible in this part:** file input/output, strings, lists, arithmetic/comparison operators, `if/else`, `for` loops, functions imported from a module, pandas cleaning, type conversion and validation.

**Project rule:** This notebook is one stage of the same Daraz analysis. It does not use separate practice examples; every concept is applied to the real project data.

## Imports and raw-data input
This part intentionally reloads the **raw CSV** so the cleaning stage is reproducible on its own. Reusable helper logic lives in `daraz_utils.py`, which is the shared Python module for this project.

In [1]:
import pandas as pd
from daraz_utils import sold_to_number

df = pd.read_csv("Watches Bags Accessories.csv")
print("Raw shape:", df.shape)

Raw shape: (1078, 11)


## Data cleaning with functions, loops, conditions and operators

> **Course concepts visible here:** functions, parameters/return values, `if / else`, `for` loops, comparison checks, string operations, lists, arithmetic operators, type conversion and file output.

From Module 5, the main problems are:

1. repeated product records need to be identified before the product link is removed
2. `Currency` is identical on every row and `Link` is only needed for identifying repeated listings
3. `Rating in Stars` is text like `4.6/5`, and `0` is used for products with no ratings
4. `Current Price` and `Original Price` are text
5. `Sold Count` is text like `6K Sold`
6. `Delivery` uses `0` to mean the listing does not show free delivery
7. `Voucher` mixes voucher counts, rupee discounts and percentage discounts

This module uses the concepts from class in a real cleaning task: a **function**, **if/else conditions**, a **for loop**, string operations and arithmetic calculations.

In [2]:
# always check missing values before doing anything else
df.isnull().sum()

Title               0
Rating in Stars     0
Rating Count        0
Sold Count         34
Voucher             0
Delivery            0
Currency            0
Current Price       0
Original Price     68
Link                0
Category            0
dtype: int64

**What this shows.** 34 products have no sold count and 68 have no original price. Every
other column is completely filled in.

In [3]:
# check duplicate rows and repeated product links before removing the Link column
print("exact duplicate rows:", df.duplicated().sum())
print("repeated product links beyond the first record:", df.duplicated(subset=["Link"]).sum())

# Link is the best product identifier available in this file.
# Keep one record per product link so the same listing is not counted more than once.
df = df.drop_duplicates(subset=["Link"], keep="first").copy()
print("rows after keeping one record per product link:", len(df))

exact duplicate rows: 14
repeated product links beyond the first record: 27
rows after keeping one record per product link: 1051


**Why we check the link before dropping it.** The file contains 14 rows that are completely identical, but there are additional records that reuse the same product URL with a small change in values such as price, rating count or sold count. The dataset has no timestamp telling us which repeated record is newest, so we use one consistent rule: keep the first record for each product link.

This leaves **1051 product records**. It prevents the same Daraz listing from being counted more than once in our category totals and comparisons.

In [4]:
# after using Link to identify repeated listings, drop columns not needed for analysis
df = df.drop(columns=["Currency", "Link"])
print("columns now:", df.shape[1])

columns now: 9


**Why we now drop these columns.** `Currency` says `Rs.` for every record, so it cannot separate or explain anything. `Link` has already served its purpose as the product identifier during duplicate checking, and we do not use URLs in the numerical analysis.

In [5]:
# rating is written as "4.6/5", so remove "/5" and convert it to a number
df["Rating in Stars"] = df["Rating in Stars"].str.replace("/5", "", regex=False)
df["Rating in Stars"] = pd.to_numeric(df["Rating in Stars"], errors="coerce")

# A rating value of 0 appears only when Rating Count is also 0.
# That means "not rated yet", not a genuine 0-star customer rating.
df.loc[df["Rating Count"] == 0, "Rating in Stars"] = None

df[["Rating in Stars", "Rating Count"]].head()

,Rating in Stars,Rating Count
0,4.6,707
1,4.3,293
2,4.5,57
3,4.6,51
4,4.4,54


In [6]:
# current price is plain text, original price has "Rs. " in front of it
df["Current Price"] = pd.to_numeric(df["Current Price"].str.replace(",", ""))

df["Original Price"] = df["Original Price"].str.replace("Rs. ", "")
df["Original Price"] = df["Original Price"].str.replace(",", "")
df["Original Price"] = pd.to_numeric(df["Original Price"], errors="coerce")

print(df["Current Price"].head(3))
print(df["Original Price"].head(3))

0    287
1    270
2    225
Name: Current Price, dtype: int64
0    520.0
1    999.0
2    750.0
Name: Original Price, dtype: float64


**How the price conversion works.** `str.replace()` removes the non-numeric text such as `Rs. ` and commas, and `pd.to_numeric()` converts the cleaned text into numbers.

We use `errors="coerce"` for `Original Price` because some records do not contain an original price. Those values remain missing rather than causing the conversion to fail or being replaced with invented data.

**Reusable sold-count function.** The conversion logic is defined once in `daraz_utils.py` as `sold_to_number()`. This is modular programming: the cleaning notebook imports and reuses the function instead of duplicating its definition in multiple notebooks.

The function uses `if` conditions, string methods, multiplication and a return value to convert `6K Sold` to `6000` and `452 Sold` to `452` while preserving missing values.

In [7]:
# run that function on every row using a loop
sold_numbers = []

for value in df["Sold Count"]:
    sold_numbers.append(sold_to_number(value))

df["Sold Count"] = sold_numbers

df["Sold Count"].head()

0    6000.0
1    1000.0
2     452.0
3     332.0
4     476.0
Name: Sold Count, dtype: float64

**How this works step by step.** We make an empty list, go through every value in the
`Sold Count` column one at a time, pass each one into our function, and add the answer to the
list. When the loop finishes, the list has one number for every row, so we put it back into
the table as the new column.

The `if pd.isna(text)` line at the top handles the 34 empty rows. Without it the function
would crash when it reached one.

In [8]:
# delivery uses 0 instead of a readable label
df["Delivery"] = df["Delivery"].replace("0", "No Free Delivery")

df["Delivery"].value_counts()

Delivery
Free Delivery       723
No Free Delivery    328
Name: count, dtype: int64

**Why we renamed this.** `0` and `Free Delivery` in the same column is difficult to interpret in a table or graph. We rename `0` to **No Free Delivery**. We do not call it “Paid Delivery” because the dataset only tells us that free delivery is not shown; it does not tell us the exact delivery charge.

In [9]:
# voucher becomes a simple yes or no
has_voucher = []

for value in df["Voucher"]:
    if str(value) == "0":
        has_voucher.append("No")
    else:
        has_voucher.append("Yes")

df["Has Voucher"] = has_voucher
df = df.drop(columns=["Voucher"])

df["Has Voucher"].value_counts()

Has Voucher
No     896
Yes    155
Name: count, dtype: int64

**Why we simplified the voucher column.** The raw inspection showed that this column mixes voucher counts, rupee amounts and percentages. There is no honest way to put those on the same numerical scale without extra rules, so for this project we reduce the field to the consistent question **Does the listing show a voucher offer: Yes or No?**

**Working out the discount ourselves.** Now that both price columns are numbers we can
calculate the discount. This was not in the original file, we are creating it.

In [10]:
# arithmetic operators create two useful variables from the cleaned price columns
# 1) discount amount in rupees
# 2) discount percentage, which is easier to compare across different price levels
df["Discount"] = df["Original Price"] - df["Current Price"]
df["Discount %"] = (df["Discount"] / df["Original Price"]) * 100

df[["Original Price", "Current Price", "Discount", "Discount %"]].head()

,Original Price,Current Price,Discount,Discount %
0,520.0,287,233.0,44.807692
1,999.0,270,729.0,72.972973
2,750.0,225,525.0,70.000000
3,299.0,160,139.0,46.488294
4,300.0,198,102.0,34.000000


In [11]:
# check that the cleaning actually worked
print(df.isnull().sum())
print()
print("rows:", df.shape[0], " columns:", df.shape[1])
df.info()

Title               0
Rating in Stars    11
Rating Count        0
Sold Count         31
Delivery            0
Current Price       0
Original Price     65
Category            0
Has Voucher         0
Discount           65
Discount %         65
dtype: int64

rows: 1051  columns: 11
<class 'pandas.core.frame.DataFrame'>
Index: 1051 entries, 0 to 1077
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Title            1051 non-null   object 
 1   Rating in Stars  1040 non-null   float64
 2   Rating Count     1051 non-null   int64  
 3   Sold Count       1020 non-null   float64
 4   Delivery         1051 non-null   object 
 5   Current Price    1051 non-null   int64  
 6   Original Price   986 non-null    float64
 7   Category         1051 non-null   object 
 8   Has Voucher      1051 non-null   object 
 9   Discount         986 non-null    float64
 10  Discount %       986 non-null    float64
dtypes: float64(5), 

**Did it work?** Yes. Columns that should be numeric are now numeric instead of text.

After keeping one record per product link, the remaining missing values come from information that was absent in the source file: some products do not have a reported sold count or original price, and 11 products have no customer rating yet. `Discount` and `Discount %` are also missing wherever the original price is missing.

We do **not** fill these missing values with guesses. Pandas can ignore missing values when calculating medians, means and correlations, which is safer than inventing data.

In [12]:
# is the original price ever lower than the current price
(df["Original Price"] < df["Current Price"]).sum()

np.int64(0)

**Why we checked this.** If the original price were lower than the current price, our calculated discount would be negative and the row would be internally inconsistent. The result is 0, so none of the available original prices produces a negative discount.

This checks internal consistency only. It does **not** independently prove that every displayed “original price” or discount on the marketplace was commercially genuine.

### File output — save the cleaned dataset

> **Course concept visible here:** file output.

After the cleaning checks pass, we export the cleaned DataFrame to a new CSV file. This is a practical use of file output: the cleaned data can be reused without repeating all raw-text cleaning steps.

In [13]:
# save the cleaned dataset as a reusable project output
df.to_csv("daraz_cleaned_data.csv", index=False)
print("Cleaned dataset saved: daraz_cleaned_data.csv")

Cleaned dataset saved: daraz_cleaned_data.csv


## Module 2 output
The main reusable output is **`daraz_cleaned_data.csv`**. Module 3 reads that file instead of repeating the cleaning code.